# T6 → T8 — xếp hạng lại ColBERT (2 GPU), đánh giá (nếu có nhãn), tạo bài nộp

**Cài đặt:** Accelerator = *GPU T4 ×2*, Internet = *On*.
**Đầu vào:** `mir-code`, `mir-data`, `mir-runs` (output của notebook T4+T5, chứa `runs/`).
**Đầu ra:** `runs/<run>_rr.parquet`, `results/`, `submissions/<tên>.zip`.

In [ ]:
# Chuẩn bị chung: chép code, đặt biến môi trường, nối (symlink) dữ liệu đầu vào vào thư mục làm việc
import os, shutil, subprocess, sys, json, glob
CODE_IN = '/kaggle/input/mir-code'          # dataset: mir/, configs/
DATA_IN = '/kaggle/input/mir-data'          # dataset: Data/ (query.parquet, links_corpus.parquet), corpus_clean/, chunks/
WORK = '/kaggle/working'
CODE = f'{WORK}/code'
if not os.path.exists(CODE):
    shutil.copytree(CODE_IN, CODE)
os.chdir(CODE)
os.environ.update(MIR_DATA_DIR=DATA_IN, MIR_WORK_DIR=WORK, HF_HUB_DISABLE_SYMLINKS_WARNING='1')
for name in ('corpus_clean', 'chunks'):
    if not os.path.exists(f'{WORK}/{name}') and os.path.exists(f'{DATA_IN}/{name}'):
        os.symlink(f'{DATA_IN}/{name}', f'{WORK}/{name}')
def sh(cmd):
    print('$', cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, f'lỗi: {cmd}'
sh('pip install -q pyyaml pyvi jieba faiss-cpu')
print(subprocess.run(f'df -h {WORK}; free -g; nproc', shell=True, capture_output=True, text=True).stdout)
CFG = 'configs/baseline.yaml'

In [ ]:
RUN = 'structure_G'          # run cần xếp hạng lại (vd cấu hình G = Dense+LSR+BM25)
os.makedirs(f'{WORK}/runs', exist_ok=True)
for p in glob.glob('/kaggle/input/mir-runs/runs/*'):
    dst = f'{WORK}/runs/{os.path.basename(p)}'
    if not os.path.exists(dst):
        shutil.copy(p, dst)
# Xếp hạng lại top-N trên 2 GPU song song (mỗi GPU một nửa số câu hỏi); chạy lại sẽ bỏ qua phần đã xong
cmd = f'python -m mir.rerank --config {CFG} --run {WORK}/runs/{RUN}.parquet'
procs = [subprocess.Popen(f'{cmd} --shard {g}/2 --device cuda:{g} > {WORK}/rerank_gpu{g}.log 2>&1', shell=True) for g in range(2)]
[p.wait() for p in procs]
for g in range(2):
    print(open(f'{WORK}/rerank_gpu{g}.log').read()[-1500:])
sh(f'{cmd} --shard 0/2 --device cuda:0')     # gộp 2 phần → runs/{RUN}_rr.parquet

In [ ]:
# Đánh giá (CHỈ khi đã có nhãn thật: đặt eval.qrels trong cấu hình hoặc truyền --qrels)
QRELS = None                 # vd '/kaggle/input/mir-qrels/qrels.parquet'
if QRELS:
    sh(f'python -m mir.evaluate --config {CFG} --qrels {QRELS} --tables')

In [ ]:
# Tạo bài nộp (kiểm tra đủ 1.200 câu, doc_id hợp lệ, chunk_text nguyên văn) → submissions/*.zip
sh(f'python -m mir.submit --config {CFG} --run {WORK}/runs/{RUN}_rr.parquet --name {RUN}_rr')
sh(f'ls -la {WORK}/submissions')